<a href="https://colab.research.google.com/github/sxsaa/Mathematical-Reasoning-in-Small-Language-Models-using-Process-Reward-Models-and-Best-of-N-Search/blob/main/Fine_tune_Qwen2_5_1_5B_Base_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl wandb

In [ ]:
import torch
import os
import wandb
import pandas as pd
from datetime import datetime
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from google.colab import userdata

In [ ]:
import os
import wandb
from google.colab import userdata
from datetime import datetime

# Log in to Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

PROJECT_NAME = "Qwen2.5-3B-Step-Verifier"
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"

# Configure Weights & Biases
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

wandb.init(project=PROJECT_NAME, name=RUN_NAME)

In [ ]:
from huggingface_hub import login

# 1. Log in to Hugging Face
hf_token = userdata.get('HF_TOKEN') # Make sure this matches your Colab secret name
login(hf_token)

In [ ]:
import torch

# Names
HUB_MODEL_NAME = f"{PROJECT_NAME.replace(' ', '-')}-{RUN_NAME}"

# Hyper-parameters - overall
EPOCHS = 1
BATCH_SIZE = 8
MAX_SEQUENCE_LENGTH = 1024
GRADIENT_ACCUMULATION_STEPS = 4 # Effective batch size = 64

# Hyper-parameters - QLoRA
QUANT_4_BIT = True
LORA_R = 64
LORA_ALPHA = LORA_R * 2
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LORA_DROPOUT = 0.1

# Hyper-parameters - training
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.03
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.01
OPTIMIZER = "paged_adamw_32bit"
WARMUP_STEPS = 500

# Hardware checks
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Tracking & Evaluation
LOG_STEPS = 10
EVAL_STEPS = 250
SAVE_STEPS = 500

### Data Subsampling Strategies

Before fine-tuning, 100,000 samples were extracted from PRM800K while maintaining a strict label ratio: **45% Correct, 45% Incorrect, 10% Neutral**. Two sampling functions were used:

1. **`smart_subsample_prm` (v1 Model)**  
   - Prioritizes the hardest math problems by "depth" (steps further into the problem).  
   - Selects long, complex steps to focus on challenging logic.

2. **`balanced_smart_subsample` (v2.0 Model)**  
   - Fixes v1 bias: uses Hard Negative Mining for tricky Incorrect steps.  
   - Uses uniform random sampling for Correct and Neutral steps to balance easy and hard examples.

> **⚠️ Training Note:** Use **only one** pre-processed dataset per fine-tuning run. Do **not** mix both datasets in the same training loop.

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd

print("Loading dataset from Hub...")
full_dataset = load_dataset("sxsaa/PRM800K-Step-Verifier", split="train")

print("Creating initial split...")
split_dataset = full_dataset.train_test_split(test_size=0.05, seed=42)

raw_train_dataset = split_dataset['train']
raw_eval_dataset = split_dataset['test']

def smart_subsample_prm(dataset, target_size=100000, max_tokens=1024):
    df = dataset.to_pandas()

    # Filter out excessively long text
    df['char_length'] = df['messages'].apply(lambda x: len(str(x)))
    df = df[df['char_length'] < (max_tokens * 4)]

    def extract_meta(msgs):
        user_prompt = msgs[0]['content']
        assistant_response = msgs[-1]['content'].strip()

        label = "Correct"
        if "Incorrect" in assistant_response: label = "Incorrect"
        elif "Neutral" in assistant_response: label = "Neutral"

        # Depth = Harder math
        depth = user_prompt.count("Step ")
        return pd.Series([label, depth])

    print(f"Analyzing step depth for {len(df)} rows...")
    df[['label', 'depth']] = df['messages'].apply(extract_meta)

    num_incorrect = int(target_size * 0.45)
    num_correct = int(target_size * 0.45)
    num_neutral = int(target_size * 0.10)

    # Priority: Grab a large pool of the deepest steps, then randomly sample to hit the exact target
    # This removes the "unique problem" bottleneck!
    pool_inc = df[df['label'] == 'Incorrect'].sort_values('depth', ascending=False).head(num_incorrect * 3)
    df_inc = pool_inc.sample(min(num_incorrect, len(pool_inc)), random_state=42)

    pool_cor = df[df['label'] == 'Correct'].sort_values('depth', ascending=False).head(num_correct * 3)
    df_cor = pool_cor.sample(min(num_correct, len(pool_cor)), random_state=42)

    pool_neu = df[df['label'] == 'Neutral'].sort_values('depth', ascending=False).head(num_neutral * 3)
    df_neu = pool_neu.sample(min(num_neutral, len(pool_neu)), random_state=42)

    final_df = pd.concat([df_inc, df_cor, df_neu]).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"\nFinal Dataset Distribution:\n{final_df['label'].value_counts()}")

    return Dataset.from_pandas(final_df[['messages']])

print("\n--- Processing Balanced Train Dataset ---")
large_train_dataset = smart_subsample_prm(raw_train_dataset, target_size=100000)

print("\n--- Processing Balanced Eval Dataset ---")
eval_dataset = smart_subsample_prm(raw_eval_dataset, target_size=1000)

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd

print("Loading dataset from Hub...")
full_dataset = load_dataset("sxsaa/PRM800K-Step-Verifier", split="train")

print("Creating initial split...")
split_dataset = full_dataset.train_test_split(test_size=0.05, seed=42)

raw_train_dataset = split_dataset['train']
raw_eval_dataset = split_dataset['test']

def balanced_smart_subsample(dataset, target_size=100000, max_tokens=1024):
    df = dataset.to_pandas()

    # 1. Filter out excessively long text
    df['char_length'] = df['messages'].apply(lambda x: len(str(x)))
    df = df[df['char_length'] < (max_tokens * 4)]

    def extract_meta(msgs):
        user_prompt = msgs[0]['content']
        assistant_response = msgs[-1]['content'].strip()

        label = "Correct"
        if "Incorrect" in assistant_response: label = "Incorrect"
        elif "Neutral" in assistant_response: label = "Neutral"

        # Depth = Harder math
        depth = user_prompt.count("Step ")
        return pd.Series([label, depth])

    print(f"Analyzing step depth for {len(df)} rows...")
    df[['label', 'depth']] = df['messages'].apply(extract_meta)

    num_incorrect = int(target_size * 0.45)
    num_correct = int(target_size * 0.45)
    num_neutral = int(target_size * 0.10)

    # --- THE CRITICAL FIX IS HERE ---

    # 1. INCORRECT DATA: Keep Hard Negative Mining (Hardest mistakes first)
    pool_inc = df[df['label'] == 'Incorrect'].sort_values('depth', ascending=False).head(num_incorrect * 3)
    df_inc = pool_inc.sample(min(num_incorrect, len(pool_inc)), random_state=42)

    # 2. CORRECT DATA: Uniform Random Sampling (Mix of easy and hard steps)
    # Notice we removed the 'sort_values' so it doesn't only pick the longest steps!
    df_cor = df[df['label'] == 'Correct'].sample(min(num_correct, len(df[df['label'] == 'Correct'])), random_state=42)

    # 3. NEUTRAL DATA: Uniform Random Sampling
    df_neu = df[df['label'] == 'Neutral'].sample(min(num_neutral, len(df[df['label'] == 'Neutral'])), random_state=42)

    # Combine and thoroughly shuffle
    final_df = pd.concat([df_inc, df_cor, df_neu]).sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\nFinal Dataset Distribution:\n{final_df['label'].value_counts()}")

    return Dataset.from_pandas(final_df[['messages']])

print("\n--- Processing Balanced Train Dataset ---")
large_train_dataset = balanced_smart_subsample(raw_train_dataset, target_size=100000)

print("\n--- Processing Balanced Eval Dataset ---")
eval_dataset = balanced_smart_subsample(raw_eval_dataset, target_size=1000)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-3B"

# Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=QUANT_4_BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Base Model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare for LoRA
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

# 1. Configure the Training Arguments
# Optimized for A100: High throughput with stable convergence
training_args = SFTConfig(
    output_dir=f"./{HUB_MODEL_NAME}",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    bf16=use_bf16,
    optim=OPTIMIZER,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_steps=WARMUP_STEPS,

    # Logging and Evaluation
    logging_strategy="steps",
    logging_steps=LOG_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    report_to="wandb",

    # Advanced Trainer Settings
    max_grad_norm=0.3,
    max_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="messages",
    completion_only_loss=True, # Focuses training loss only on the assistant's label
)

# 2. Setup the Trainer
# The trainer will automatically use the peft_config to wrap the base model
trainer = SFTTrainer(
    model=model,
    train_dataset=large_train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
)

# 3. Execute Training
print(f"Starting Training on {len(large_train_dataset)} samples...")
trainer.train()

# 4. Finish Logging
wandb.finish()

# 5. Save the final LoRA adapter
print("Saving final model and tokenizer...")
trainer.model.save_pretrained(f"./{HUB_MODEL_NAME}-final")
tokenizer.save_pretrained(f"./{HUB_MODEL_NAME}-final")

In [ ]:
from peft import PeftModel
import torch

print("Merging LoRA with Base Model... This requires high RAM.")

# 1. Load the base model again (unquantized for merging)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    device_map="cpu" # Use CPU to save VRAM for the merge
)

# 2. Load your fine-tuned adapter
model_to_merge = PeftModel.from_pretrained(base_model, f"./{HUB_MODEL_NAME}-final")

# 3. Merge and Unload
merged_model = model_to_merge.merge_and_unload()

# 4. Push the full 3B model to HF
merged_repo_id = f"sxsaa/Qwen2.5-3B-Math-Verifier-FullData-v2.0"
merged_model.push_to_hub(merged_repo_id)
tokenizer.push_to_hub(merged_repo_id)

print(f"✅ Full merged model is live at {merged_repo_id}")